In [3]:
# ============================================================
# Mean-Variance-Skewness Portfolio Optimization
# ============================================================
# Description:
# Portfolio optimization using Higher Order Moments
# Objective:
# Maximize:
# alpha * Expected Return
# - beta * Variance
# + gamma * Skewness
# ============================================================

# =========================
# IMPORTS
# =========================
import yfinance as yf
import pandas as pd
import numpy as np

from scipy.optimize import minimize
from scipy.stats import skew

# =========================
# ASSETS
# =========================
assets = [
    "RELIANCE.NS",
    "TCS.NS",
    "HDFCBANK.NS",
    "INFY.NS",
    "ICICIBANK.NS",
    "SBIN.NS",
    "LT.NS",
    "ITC.NS",
    "BHARTIARTL.NS",
    "GOLDBEES.NS"
]

# =========================
# DOWNLOAD DATA
# =========================
prices = yf.download(
    assets,
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

# =========================
# DAILY RETURNS
# =========================
returns = prices.pct_change().dropna()

# =========================
# STATISTICS
# =========================
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

num_assets = len(assets)

# =========================
# PORTFOLIO FUNCTIONS
# =========================
def portfolio_return(weights):
    return np.dot(weights, mean_returns)


def portfolio_variance(weights):
    return np.dot(weights.T,
                  np.dot(cov_matrix, weights))


def portfolio_skewness(weights):
    portfolio_returns = returns.dot(weights)
    return skew(portfolio_returns)


# =========================
# OBJECTIVE FUNCTION
# =========================
alpha = 1.0      # Reward Return
beta = 3.0       # Penalize Variance
gamma = 1.0      # Reward Positive Skewness


def objective(weights):

    ret = portfolio_return(weights)
    var = portfolio_variance(weights)
    sk = portfolio_skewness(weights)

    utility = (
        alpha * ret
        - beta * var
        + gamma * sk
    )

    return -utility


# =========================
# # =========================
# CONSTRAINTS
# =========================

# Sum of weights = 1
constraints = (
    {
        'type': 'eq',
        'fun': lambda w: np.sum(w) - 1
    },
)

# Minimum 5% and Maximum 30% per asset
bounds = tuple((0.05, 0.30) for _ in range(num_assets))

# Initial equal weights
initial_weights = np.ones(num_assets) / num_assets

# =========================
# OPTIMIZATION
# =========================

result = minimize(
    objective,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

optimal_weights = result.x

# =========================
# FINAL PORTFOLIO
# =========================

final_return = portfolio_return(optimal_weights)
final_variance = portfolio_variance(optimal_weights)
final_volatility = np.sqrt(final_variance)
final_skewness = portfolio_skewness(optimal_weights)
# =========================
# =========================
# OUTPUT
# =========================
weights_df = pd.DataFrame({
    "Asset": assets,
    "Weight": optimal_weights
})

weights_df["Weight"] = weights_df["Weight"].round(4)

print("=" * 60)
print("MEAN-VARIANCE-SKEWNESS PORTFOLIO")
print("=" * 60)

print("\nOptimal Weights")
print(weights_df)

print("\nPortfolio Statistics")
print("-" * 60)

print(f"Expected Annual Return : {final_return:.4f}")
print(f"Annual Variance        : {final_variance:.4f}")
print(f"Annual Volatility      : {final_volatility:.4f}")
print(f"Portfolio Skewness     : {final_skewness:.4f}")

print("\nObjective Function Value")
print(-result.fun)

print("\nOptimization Success:", result.success)
print("Message:", result.message)

# =========================
# SAVE RESULTS
# =========================
weights_df.to_csv(
    "Mean_Variance_Skewness_Portfolio.csv",
    index=False
)

print("\nPortfolio saved as:")
print("Mean_Variance_Skewness_Portfolio.csv")

MEAN-VARIANCE-SKEWNESS PORTFOLIO

Optimal Weights
           Asset  Weight
0    RELIANCE.NS    0.30
1         TCS.NS    0.05
2    HDFCBANK.NS    0.05
3        INFY.NS    0.05
4   ICICIBANK.NS    0.05
5        SBIN.NS    0.05
6          LT.NS    0.05
7         ITC.NS    0.30
8  BHARTIARTL.NS    0.05
9    GOLDBEES.NS    0.05

Portfolio Statistics
------------------------------------------------------------
Expected Annual Return : 0.2223
Annual Variance        : 0.0379
Annual Volatility      : 0.1946
Portfolio Skewness     : -0.6803

Objective Function Value
-0.5716071275815142

Optimization Success: True
Message: Optimization terminated successfully

Portfolio saved as:
Mean_Variance_Skewness_Portfolio.csv
